# Pandas — Phase 7: Reshaping, Binning & Advanced GroupBy
### Credit Card Risk Analysis Track

**Topics in this phase:**
29. Reshaping Data — `melt()`, `stack()`/`unstack()`, `pivot()`
30. Binning Continuous Variables — `pd.cut()`, `pd.qcut()`
31. GroupBy `.transform()` and `.apply()`
32. The `.query()` Method

**Dataset:** the clean `loan_applications.csv` from Phase 1. Keep it in the same folder as this notebook.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual dataset before this notebook was assembled.
- Like Phase 4-6, later tasks build on columns created by earlier ones — run top to bottom.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])
df["annual_income"] = df["annual_income"].fillna(df["annual_income"].median())
df["credit_score"] = df["credit_score"].fillna(df["credit_score"].median())
print(df.shape)
df.head()

## Topic 29: Reshaping Data

So far every column has stayed a column — "wide" format. Reshaping turns wide data long (or back again), which many plotting and modeling tools actually expect.

**Q1.** Use `df.melt()` to reshape three financial columns (`annual_income`, `loan_amount`, `existing_debt`) into **long** format: one row per `(application_id, metric, amount)` combination. Keep `application_id` as `id_vars`, put the three financial columns in `value_vars`, and name the resulting columns `metric` and `amount` via `var_name`/`value_name`. Store as `long_financials` and print its shape (it should have 3× the rows of `df`, since each application now spans 3 rows instead of 1).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
long_financials = df.melt(
    id_vars=["application_id"],
    value_vars=["annual_income", "loan_amount", "existing_debt"],
    var_name="metric",
    value_name="amount",
)
print(long_financials.shape)
print(long_financials.head(3))

**Q2.** `.pivot()` is `melt`'s inverse — it only works when each `(index, columns)` combination is unique (no aggregation happens, unlike `pivot_table`). Given `small_wide` below (already in long format), reshape it back to wide using `.pivot(index="applicant", columns="metric", values="amount")`, into `pivoted_back`.

In [ ]:
small_wide = pd.DataFrame({
    "applicant": ["A101", "A101", "A102", "A102"],
    "metric": ["income", "loan", "income", "loan"],
    "amount": [50000, 12000, 60000, 8500],
})

# YOUR CODE HERE


**Solution**

In [ ]:
pivoted_back = small_wide.pivot(index="applicant", columns="metric", values="amount")
print(pivoted_back)

**Q3.** `.stack()` compresses a DataFrame's **columns** down into an extra index level, turning it into a Series. Take the first 3 rows of `application_id`-indexed `annual_income`/`loan_amount`/`existing_debt` (`subset`, given below), and call `.stack()` on it into `stacked`. Print it — notice the result is a Series with a two-level index (application_id, column name).

In [ ]:
subset = df.set_index("application_id")[["annual_income", "loan_amount", "existing_debt"]].head(3)

# YOUR CODE HERE


**Solution**

In [ ]:
stacked = subset.stack()
print(stacked)

**Q4.** `.unstack()` reverses `.stack()` — it takes the innermost index level and turns it back into columns. Call `.unstack()` on `stacked` from Q3, into `unstacked`, and confirm with `.equals()` that you're back to something identical to `subset`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
unstacked = stacked.unstack()
print(unstacked.equals(subset))

**Q5.** This is where `.unstack()` earns its keep on real analysis: a two-key `.groupby()` (from Phase 5) gives you a long Series with a 2-level index — often awkward to read. Group `df` by `employment_status` and `home_ownership`, take the mean `loan_amount`, into `grouped_long`. Then `.unstack()` it into `grouped_wide` — the second grouping key becomes columns, turning it into a clean spreadsheet-style table (equivalent to what `pivot_table` would have given you directly, but built from a groupby you already know how to do).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
grouped_long = df.groupby(["employment_status", "home_ownership"])["loan_amount"].mean()
grouped_wide = grouped_long.unstack()
print(grouped_wide.round(0))

**Q6.** Redo a melt like Q1, but this time also keep `loan_status` as an extra `id_vars` column (so it rides along on every melted row), melting only `annual_income` and `loan_amount`, with columns named `field_name`/`field_value`, into `long_custom`. Print the columns and shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
long_custom = df.melt(
    id_vars=["application_id", "loan_status"],
    value_vars=["annual_income", "loan_amount"],
    var_name="field_name",
    value_name="field_value",
)
print(long_custom.columns.tolist(), long_custom.shape)

## Topic 30: Binning Continuous Variables

You built risk tiers by hand with `np.select`/`np.digitize` back in the numpy phase. Pandas has purpose-built tools for exactly this.

**Q7.** Use `pd.cut(df["credit_score"], bins=5)` to split `credit_score` into 5 **equal-width** bins (pandas picks the bin edges automatically, splitting the observed range into 5 equal-size intervals), into a new column `credit_score_bin`. Print the value counts, sorted by bin order.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["credit_score_bin"] = pd.cut(df["credit_score"], bins=5)
print(df["credit_score_bin"].value_counts().sort_index())

**Q8.** Equal-width bins rarely match real risk-tier business rules. Use `pd.cut` with **explicit edges** `[0, 580, 670, 740, 850]` and matching `labels=["Subprime", "Near-Prime", "Prime", "Super-Prime"]` into a new `risk_tier` column — this is the same tiering logic you built with `np.select` before, but in one line. Print the value counts.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["risk_tier"] = pd.cut(
    df["credit_score"],
    bins=[0, 580, 670, 740, 850],
    labels=["Subprime", "Near-Prime", "Prime", "Super-Prime"],
)
print(df["risk_tier"].value_counts())

**Q9.** `pd.qcut` bins by **quantile** instead of by value — every bin gets (roughly) the same *number* of rows, regardless of the value range each bin ends up covering. Use `pd.qcut(df["annual_income"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])` into `income_quartile`. Print the value counts — they should be much closer to equal than Q7's equal-width bins.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["income_quartile"] = pd.qcut(df["annual_income"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df["income_quartile"].value_counts())

**Q10.** Get `bin_distribution`: the value counts of `risk_tier` from Q8, sorted by the tier order (not by count) using `.sort_index()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
bin_distribution = df["risk_tier"].value_counts().sort_index()
print(bin_distribution)

**Q11.** Compute `avg_loan_by_tier`: mean `loan_amount` grouped by the `risk_tier` bins from Q8. Pass `observed=True` to `.groupby()` — `pd.cut` produces a special "categorical" dtype, and `observed=True` tells pandas to only show tiers that actually appear in the data rather than every possible category.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
avg_loan_by_tier = df.groupby("risk_tier", observed=True)["loan_amount"].mean()
print(avg_loan_by_tier)

**Q12.** By default, `pd.cut` bins are **right-closed** — `(580, 670]` includes 670 but not 580. Redo Q8's cut with `right=False` instead, into `risk_tier_leftclosed` — now bins are left-closed, `[580, 670)`, including 580 but not 670. Compare it to `risk_tier` from Q8 and count how many rows land in a **different** tier because of this edge-case difference — a handful of borrowers sitting exactly on a threshold value are the ones affected.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["risk_tier_leftclosed"] = pd.cut(
    df["credit_score"],
    bins=[0, 580, 670, 740, 850],
    labels=["Subprime", "Near-Prime", "Prime", "Super-Prime"],
    right=False,
)
print((df["risk_tier"] != df["risk_tier_leftclosed"]).sum())

## Topic 31: GroupBy `.transform()` and `.apply()`

`.agg()` collapses a group to one row. `.transform()` and `.apply()` are for when you need something back that isn't a simple collapse.

**Q13.** `.transform()` returns a result the **same length as the original DataFrame** — perfect for adding a new column without a separate merge step. Add `loan_amount_zscore_within_tier`: for each row, how many standard deviations its `loan_amount` is from its **own risk tier's** mean (not the whole portfolio's), using `.groupby("risk_tier", observed=True)["loan_amount"].transform(lambda x: (x - x.mean()) / x.std())`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["loan_amount_zscore_within_tier"] = df.groupby("risk_tier", observed=True)["loan_amount"].transform(
    lambda x: (x - x.mean()) / x.std()
)
print(df["loan_amount_zscore_within_tier"].head(3).tolist())

**Q14.** Add `peer_avg_income`: for every row, the average `annual_income` of everyone sharing that row's `employment_status` — a "how does this person compare to their peer group" feature, using `.groupby("employment_status")["annual_income"].transform("mean")`. Print `employment_status`, `annual_income`, and `peer_avg_income` side by side for a few rows.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["peer_avg_income"] = df.groupby("employment_status")["annual_income"].transform("mean")
print(df[["employment_status", "annual_income", "peer_avg_income"]].head(3))

**Q15.** `.apply()` is more flexible than `.transform()` — the function you give it can return whatever shape it wants, including something that collapses each group to one value (like `.agg()` would) but via custom logic `.agg()` can't express directly. Compute `corr_by_employment`: the correlation between `loan_amount` and `credit_score`, calculated **separately for each** `employment_status` group, using `.groupby("employment_status").apply(lambda g: g["loan_amount"].corr(g["credit_score"]), include_groups=False)`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
corr_by_employment = df.groupby("employment_status").apply(
    lambda g: g["loan_amount"].corr(g["credit_score"]), include_groups=False
)
print(corr_by_employment)

**Q16.** `.apply()` can also return a **whole sub-table** per group, not just a scalar. Get the top 2 largest loans **within each** `risk_tier`, using `.groupby("risk_tier", observed=True).apply(lambda g: g.nlargest(2, "loan_amount"))`, into `top2_per_tier`. Print just the `loan_amount` column of the result — notice the index now has **two levels**: the risk tier, and the original row index within that tier.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
top2_per_tier = df.groupby("risk_tier", observed=True).apply(lambda g: g.nlargest(2, "loan_amount"))
print(top2_per_tier[["loan_amount"]])

**Q17.** Confirm the core difference directly: check that `loan_amount_zscore_within_tier` (from `.transform()`, Q13) has exactly `len(df)` values (`transform_len_match`), while `top2_per_tier` (from `.apply()`, Q16) does **not** have `len(df)` rows (`apply_len_differs`) — `.transform()` always comes back the same shape as what went in; `.apply()` doesn't have to.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
transform_len_match = len(df["loan_amount_zscore_within_tier"]) == len(df)
apply_len_differs = len(top2_per_tier) != len(df)
print(transform_len_match, apply_len_differs)

**Q18.** Build a genuinely custom multi-column group metric with `.apply()`: a function `risk_metric(group)` that returns a `pd.Series` with three values — `avg_loan` (mean `loan_amount`), `avg_score` (mean `credit_score`), and `loan_to_score_ratio` (`avg_loan / avg_score`) — then apply it per `risk_tier` (`include_groups=False`) into `custom_metric_by_tier`. This is the kind of composite metric `.agg()` alone can't build in one step, because it depends on **two different columns combined**, not just one column's stat.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
def risk_metric(group):
    return pd.Series({
        "avg_loan": group["loan_amount"].mean(),
        "avg_score": group["credit_score"].mean(),
        "loan_to_score_ratio": group["loan_amount"].mean() / group["credit_score"].mean(),
    })

custom_metric_by_tier = df.groupby("risk_tier", observed=True).apply(risk_metric, include_groups=False)
print(custom_metric_by_tier)

## Topic 32: The `.query()` Method

`.query()` lets you write a filter condition as a **string expression** instead of a boolean mask — often more readable, especially with several conditions.

**Q19.** Rewrite `df[df["credit_score"] < 600]` using `.query()` instead, into `low_credit_query`: `df.query("credit_score < 600")`. Print the shape and confirm it matches what you'd expect from Phase 2.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
low_credit_query = df.query("credit_score < 600")
print(low_credit_query.shape)

**Q20.** Combine two conditions in one query string: `annual_income < 30000` **and** `loan_amount > 10000`, into `multi_cond_query`. Notice `.query()` lets you write `and`/`or` as plain English keywords, unlike boolean masks which need `&`/`|`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
multi_cond_query = df.query("annual_income < 30000 and loan_amount > 10000")
print(multi_cond_query.shape)

**Q21.** Given a Python variable `threshold = 700`, filter to `credit_score >= threshold` inside a query string using the `@` prefix to reference it: `df.query("credit_score >= @threshold")`, into `above_threshold_query`. This is how `.query()` reaches outside the string into your regular Python variables.

In [ ]:
threshold = 700

# YOUR CODE HERE


**Solution**

In [ ]:
above_threshold_query = df.query("credit_score >= @threshold")
print(above_threshold_query.shape)

**Q22.** Given `purposes_of_interest = ["Approved", "Pending"]`, filter to `loan_status` being one of those values using `df.query("loan_status in @purposes_of_interest")`, into `status_query` — `.query()` supports `in` directly, the string equivalent of `.isin()`.

In [ ]:
purposes_of_interest = ["Approved", "Pending"]

# YOUR CODE HERE


**Solution**

In [ ]:
status_query = df.query("loan_status in @purposes_of_interest")
print(status_query.shape)

**Q23.** Prove `.query()` and boolean masking produce identical results: rebuild Q20's filter with a plain boolean mask (`mask_version`), then compare it to `multi_cond_query` with `.equals()`. `.query()` is a readability choice, not a different filtering mechanism underneath.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
mask_version = df[(df["annual_income"] < 30000) & (df["loan_amount"] > 10000)]
query_matches_mask = mask_version.equals(multi_cond_query)
print(query_matches_mask)

## ✅ Checkpoint

**What you covered:**
- Reshaping: `melt()` (wide→long), `pivot()` (long→wide, no aggregation), `stack()`/`unstack()` (moving between columns and index levels), and unstacking a two-key groupby into a wide table
- Binning: `pd.cut()` for equal-width or custom-edge bins with labels, `pd.qcut()` for equal-frequency (quantile) bins, `observed=True` for categorical groupbys, and the `right=` edge-inclusion gotcha
- `.transform()` vs `.apply()`: same-length output for adding columns (peer averages, within-group z-scores) vs. flexible output for custom scalars, correlations, or whole sub-tables per group
- `.query()`: string-based filtering with `and`/`or`/`in`, referencing external variables with `@`, and confirming it's equivalent to boolean masking

**Why it matters for the project:** reshaping is what most plotting libraries and some modeling pipelines actually expect as input; binning is the standard way risk tiers get built in practice; `.transform()`-based peer features (like `peer_avg_income`) are a genuinely common feature engineering pattern; and `.query()` becomes valuable the moment your filter conditions get long enough that a boolean mask turns into an unreadable wall of parentheses.

**What's next:** Phase 8 — time series tools, string manipulation, outlier detection, exporting, and MultiIndex basics.